# 📧 SMS Spam Detection — NLP Pipeline

A complete **Natural Language Processing pipeline** for classifying SMS messages as **Spam** or **Ham (legitimate)** using text preprocessing, linguistic analysis, and a Naive Bayes classifier.

---

## 🗂️ Table of Contents

| # | Section |
|---|---------|
| 1 | [Setup & Data Loading](#setup) |
| 2 | [Exploratory Data Analysis](#eda) |
| 3 | [Text Preprocessing](#preprocessing) |
| 4 | [POS Tagging & NER](#linguistic) |
| 5 | [Vectorization & Splitting](#vectorization) |
| 6 | [Model Training](#training) |
| 7 | [Evaluation](#evaluation) |
| 8 | [Results & Inference](#results) |

> **Dataset:** SMS Spam Collection — 5,572 labeled messages (747 spam, 4,825 ham)  
> **Model:** Multinomial Naive Bayes · **Vectorizer:** Binary Bag-of-Words  
> **Libraries:** `pandas` · `nltk` · `spaCy` · `scikit-learn` · `plotly`


---

## 📦 Section 1 — Setup & Data Loading <a name="setup"></a>

### 1.1 Import Libraries

We import all required libraries upfront:

| Library | Purpose |
|---------|---------|
| `pandas` | Data manipulation and DataFrame operations |
| `plotly.express` | Interactive visualizations |
| `string` / `re` | Text cleaning utilities |
| `nltk` | Tokenization, stop words, lemmatization |
| `spaCy` | POS tagging and Named Entity Recognition |


In [1]:
# Importing Liberaries
import pandas as pd
import plotly.express as px
import string
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
!python -m spacy download en_core_web_sm

[nltk_data] Downloading package punkt to C:\Users\DELL
[nltk_data]     7530\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\DELL
[nltk_data]     7530\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\DELL
[nltk_data]     7530\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\DELL
[nltk_data]     7530\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 426.7 kB/s eta 0:00:29
     - ------------------------------------- 0.5/12.8 MB 426.7 kB/s eta 0:00:29
     - ------------------------------------- 0.5/12.8 MB 426.7 kB/s eta 0:00:29
     -- ------------------------------------ 0.8/12.8 MB 463.1 kB/s eta 0:00:26
     -- ------------------------------------ 0.8/12.8 MB 463.1 kB/s eta 0:00:26
     --- ----------------------------------- 1.0/12.8 MB 497.3 kB/s eta 0:00:24
     --- ----------------------------------- 1.0/12.8 MB 497.3 kB/s eta 0:00:24
     ---


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1.2 Load the Dataset

We load `spam.csv` using `latin-1` encoding (required for special characters in this dataset). The raw file contains several unnamed columns — we will keep only the two relevant ones.


In [2]:
# reading data
data = pd.read_csv(r"spam.csv", encoding="latin-1")
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


### 1.3 Select & Rename Columns

We retain only `v1` (label) and `v2` (message text), renaming them to `label` and `message` for clarity. Labels are either `"ham"` (legitimate) or `"spam"`.


In [3]:
pd.set_option('display.max_colwidth', None)

data= data [['v1','v2']]

data.columns = ['label', 'message']

data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


---

## 📊 Section 2 — Exploratory Data Analysis <a name="eda"></a>

### 2.1 Class Distribution

Understanding class imbalance is critical before modeling. The dataset is **heavily skewed** — ham messages outnumber spam roughly **6.5 : 1**. This means:

- The model must be particularly sensitive to **spam recall** — missing a spam is more costly than a false alarm
- We use **stratified splitting** later to preserve this ratio in both train and test sets


In [4]:
label_df=data["label"].value_counts().reset_index()
label_df

,label,count
0,ham,4825
1,spam,747


In [5]:
fig = px.bar(
    label_df,
    x="label",
    y="count",
    title="Ham Vs Spam",
    color_discrete_sequence=px.colors.qualitative.Bold,
    text_auto=True

)
fig.update_traces(textposition="outside")
fig.update_layout( width=600 ,  height=500  )

fig.show()

---

## 🔧 Section 3 — Text Preprocessing <a name="preprocessing"></a>

Raw SMS messages contain a lot of noise. We apply a **sequential 5-step pipeline** to produce clean, model-ready text:

```
Raw Message
    │
    ▼  [1] Punctuation Removal   — keep $ and ! as spam signals
    │
    ▼  [2] Lowercasing           — unify case variants
    │
    ▼  [3] Tokenization          — split into word tokens
    │
    ▼  [4] Stop Word Removal     — drop filler words
    │
    ▼  [5] Lemmatization         — reduce words to base form
    │
    ▼
Cleaned Text
```


### 3.1 Punctuation Removal

We strip most punctuation to reduce vocabulary noise. However, we **intentionally keep `$` and `!`** because these symbols are strong spam indicators:

> *"Win $1000 NOW!!!"* — the `$` and `!` are discriminative features we don't want to lose.


In [6]:
def remove_punctuation(text):

    keep_symbols = ['$', '!']    # علامات الترقيم اللي عايزين نحافظ عليهالانها هتميز الاسبام
    punctuationfree = "".join([i for i in text if i not in string.punctuation or i in keep_symbols])
    return punctuationfree

data['punctuation_free_message'] = data['message'].apply(remove_punctuation)

### 3.2 Lowercasing

We convert all characters to lowercase so that `"FREE"`, `"Free"`, and `"free"` are treated as the same token. This reduces vocabulary size and prevents duplicate features.


In [7]:
data['lower_message']= data['punctuation_free_message'].apply(lambda x: x.lower())

### 3.3 Tokenization

We split each message into a list of individual **tokens** using NLTK's `word_tokenize`. Tokenization is the foundation for all subsequent steps — we process language token by token.

> **Example:** `"win free prize now"` → `["win", "free", "prize", "now"]`


In [8]:
from nltk.tokenize import word_tokenize

def tokenization(text):
    return word_tokenize(text)

data['tokenized_message'] = data['lower_message'].apply(tokenization)

### 3.4 Stop Word Removal

We remove common English **stop words** (e.g., `"the"`, `"is"`, `"in"`) that carry little semantic meaning. This reduces noise and allows the model to focus on content-bearing words.

> NLTK's English stop word list contains **~179 words** such as: *"i", "me", "my", "myself", "we", "our"...*


In [9]:
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    output = [i for i in text if i not in stop_words]
    return output

data['no_stopwords_message'] = data['tokenized_message'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to C:\Users\DELL
[nltk_data]     7530\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 3.5 Lemmatization

We reduce each token to its **base dictionary form** (lemma) using NLTK's `WordNetLemmatizer`. This groups morphological variants so the model sees fewer, more meaningful features.

> **Examples:** `"winning"` → `"win"` | `"prizes"` → `"prize"` | `"claimed"` → `"claim"`

Lemmatization is preferred over **stemming** here because it always produces valid English words, which improves interpretability.


In [10]:
from nltk.stem import WordNetLemmatizer

wordnet_lemmatizer = WordNetLemmatizer()

def lemmatizer(text):
    lemm_text = [wordnet_lemmatizer.lemmatize(word) for word in text]
    return lemm_text

data['lemmatized_message']=data['no_stopwords_message'].apply(lemmatizer)

data.head()

,label,message,punctuation_free_message,lower_message,tokenized_message,no_stopwords_message,lemmatized_message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat,go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat,"[go, until, jurong, point, crazy, available, only, in, bugis, n, great, world, la, e, buffet, cine, there, got, amore, wat]","[go, jurong, point, crazy, available, bugis, n, great, world, la, e, buffet, cine, got, amore, wat]","[go, jurong, point, crazy, available, bugis, n, great, world, la, e, buffet, cine, got, amore, wat]"
1,ham,Ok lar... Joking wif u oni...,Ok lar Joking wif u oni,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005 Text FA to 87121 to receive entry questionstd txt rateTCs apply 08452810075over18s,free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s,"[free, entry, in, 2, a, wkly, comp, to, win, fa, cup, final, tkts, 21st, may, 2005, text, fa, to, 87121, to, receive, entry, questionstd, txt, ratetcs, apply, 08452810075over18s]","[free, entry, 2, wkly, comp, win, fa, cup, final, tkts, 21st, may, 2005, text, fa, 87121, receive, entry, questionstd, txt, ratetcs, apply, 08452810075over18s]","[free, entry, 2, wkly, comp, win, fa, cup, final, tkts, 21st, may, 2005, text, fa, 87121, receive, entry, questionstd, txt, ratetcs, apply, 08452810075over18s]"
3,ham,U dun say so early hor... U c already then say...,U dun say so early hor U c already then say,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, then, say]","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, early, hor, u, c, already, say]"
4,ham,"Nah I don't think he goes to usf, he lives around here though",Nah I dont think he goes to usf he lives around here though,nah i dont think he goes to usf he lives around here though,"[nah, i, dont, think, he, goes, to, usf, he, lives, around, here, though]","[nah, dont, think, goes, usf, lives, around, though]","[nah, dont, think, go, usf, life, around, though]"


### 3.6 Reassemble & Normalize

After lemmatization, tokens are **joined back** into a single string and passed through a **normalization function** that fixes common SMS abbreviations. This handles informal language that the steps above cannot address.

| Abbreviation | Replacement | Rationale |
|---|---|---|
| `" n "` | `" and "` | Very common SMS shorthand |
| `" wat "` | `" what "` | Phonetic spelling |
| `" la "` | `" "` | Filler particle — no semantic value |


In [11]:
data['cleaned_text'] = data['lemmatized_message'].apply(lambda x: " ".join(x))
data=data[['label', 'cleaned_text']]
data.head()


,label,cleaned_text
0,ham,go jurong point crazy available bugis n great world la e buffet cine got amore wat
1,ham,ok lar joking wif u oni
2,spam,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s
3,ham,u dun say early hor u c already say
4,ham,nah dont think go usf life around though


In [12]:
def normalize_text(text):
    text = text.replace(' n ', ' and ')
    text = text.replace(' wat ', ' what ')
    text = text.replace(' la ', ' ')
    return text

data['normalized_text'] = data['cleaned_text'].apply(normalize_text)

data[['cleaned_text', 'normalized_text']].head()

,cleaned_text,normalized_text
0,go jurong point crazy available bugis n great world la e buffet cine got amore wat,go jurong point crazy available bugis and great world e buffet cine got amore wat
1,ok lar joking wif u oni,ok lar joking wif u oni
2,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s
3,u dun say early hor u c already say,u dun say early hor u c already say
4,nah dont think go usf life around though,nah dont think go usf life around though


---

## 🏷️ Section 4 — Linguistic Feature Extraction <a name="linguistic"></a>

Beyond simple bag-of-words features, we extract **grammatical and semantic structure** from each message using two advanced NLP techniques.

---

### 4.1 Part-of-Speech (POS) Tagging

**POS tagging** assigns a grammatical role to every word — noun, verb, adjective, etc. — revealing the syntactic structure of messages.

**Why spaCy instead of NLTK for POS tagging?**

| Feature | NLTK | spaCy |
|---------|------|-------|
| Accuracy on informal text | Moderate | High ✅ |
| Slang & abbreviation handling | Limited | Better ✅ |
| Processing speed | Slower | Faster ✅ |
| Context-aware tagging | No | Yes ✅ |

spaCy uses neural models that understand word context, making it significantly more accurate on the short, informal, abbreviation-heavy style of SMS text.


#### Install & Load spaCy

We use the `en_core_web_sm` model — a lightweight English pipeline (~12 MB) that includes a **POS tagger**, **dependency parser**, and **NER component**.


In [15]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import spacy

nlp = spacy.load("en_core_web_sm")

#### Apply POS Tagging

We run each cleaned message through the spaCy pipeline and extract `(word, POS_tag)` pairs for every token.


In [14]:
def apply_pos_tagging(text):
    doc = nlp(text)
    return [(token.text, token.pos_) for token in doc]

data['pos_tags'] = data['cleaned_text'].apply(apply_pos_tagging)

data.head()

,label,cleaned_text,normalized_text,pos_tags
0,ham,go jurong point crazy available bugis n great world la e buffet cine got amore wat,go jurong point crazy available bugis and great world e buffet cine got amore wat,"[(go, VERB), (jurong, PROPN), (point, NOUN), (crazy, PROPN), (available, ADJ), (bugis, PROPN), (n, ADP), (great, ADJ), (world, NOUN), (la, PROPN), (e, PROPN), (buffet, PROPN), (cine, PROPN), (got, VERB), (amore, ADV), (wat, NOUN)]"
1,ham,ok lar joking wif u oni,ok lar joking wif u oni,"[(ok, INTJ), (lar, ADJ), (joking, NOUN), (wif, NOUN), (u, PRON), (oni, NOUN)]"
2,spam,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,"[(free, ADJ), (entry, NOUN), (2, NUM), (wkly, ADJ), (comp, NOUN), (win, NOUN), (fa, PROPN), (cup, PROPN), (final, ADJ), (tkts, NOUN), (21st, NOUN), (may, AUX), (2005, NUM), (text, NOUN), (fa, NOUN), (87121, NUM), (receive, VERB), (entry, NOUN), (questionstd, NOUN), (txt, VERB), (ratetcs, NOUN), (apply, VERB), (08452810075over18s, PROPN)]"
3,ham,u dun say early hor u c already say,u dun say early hor u c already say,"[(u, PRON), (dun, PROPN), (say, VERB), (early, ADJ), (hor, NOUN), (u, NOUN), (c, PROPN), (already, ADV), (say, VERB)]"
4,ham,nah dont think go usf life around though,nah dont think go usf life around though,"[(nah, INTJ), (do, AUX), (nt, PART), (think, VERB), (go, VERB), (usf, ADJ), (life, NOUN), (around, ADP), (though, ADV)]"


#### Sample Output Inspection

Let's examine how the tagger handles a real message from our dataset to verify correctness.


In [16]:
print("Cleaned Text:\n", data['cleaned_text'][0])
print("\nPOS Tags:\n", data['pos_tags'][0])

Cleaned Text:
 go jurong point crazy available bugis n great world la e buffet cine got amore wat

POS Tags:
 [('go', 'VERB'), ('jurong', 'PROPN'), ('point', 'NOUN'), ('crazy', 'PROPN'), ('available', 'ADJ'), ('bugis', 'PROPN'), ('n', 'ADP'), ('great', 'ADJ'), ('world', 'NOUN'), ('la', 'PROPN'), ('e', 'PROPN'), ('buffet', 'PROPN'), ('cine', 'PROPN'), ('got', 'VERB'), ('amore', 'ADV'), ('wat', 'NOUN')]


#### POS Tag Distribution

We aggregate all POS tags across the dataset to understand which grammatical categories dominate. Spam messages tend to use more **proper nouns (PROPN)**, **numbers (NUM)**, and **imperative verbs (VERB)** compared to ham.


In [17]:
from collections import Counter

all_tags = []

for tags in data['pos_tags']:
    for word, tag in tags:
        all_tags.append(tag)

tag_counts = Counter(all_tags)

tag_counts.most_common(10)

[('NOUN', 17030),
 ('VERB', 10573),
 ('PROPN', 7875),
 ('ADJ', 5141),
 ('NUM', 3091),
 ('ADV', 2623),
 ('PRON', 2098),
 ('INTJ', 1726),
 ('AUX', 1682),
 ('PUNCT', 1429)]

#### Visualize Top POS Tags


In [18]:
import plotly.express as px
import pandas as pd

tag_df = pd.DataFrame(tag_counts.most_common(10), columns=['POS Tag', 'Count'])

fig = px.bar(
    tag_df,
    x='POS Tag',
    y='Count',
    title='Top POS Tags in Dataset (spaCy)',
    color_discrete_sequence=px.colors.qualitative.Bold,
    text_auto=True
)

fig.show()

#### Feature Engineering from POS Tags

We convert POS tags into **numerical frequency vectors** using `Counter`. This creates a structured feature dictionary that can complement or replace raw text features in downstream models.


In [19]:
def extract_pos_features(tagged_sentence):
    tags = [tag for word, tag in tagged_sentence]
    return Counter(tags)

data['pos_features'] = data['pos_tags'].apply(extract_pos_features)

data.head()

,label,cleaned_text,normalized_text,pos_tags,pos_features
0,ham,go jurong point crazy available bugis n great world la e buffet cine got amore wat,go jurong point crazy available bugis and great world e buffet cine got amore wat,"[(go, VERB), (jurong, PROPN), (point, NOUN), (crazy, PROPN), (available, ADJ), (bugis, PROPN), (n, ADP), (great, ADJ), (world, NOUN), (la, PROPN), (e, PROPN), (buffet, PROPN), (cine, PROPN), (got, VERB), (amore, ADV), (wat, NOUN)]","{'VERB': 2, 'PROPN': 7, 'NOUN': 3, 'ADJ': 2, 'ADP': 1, 'ADV': 1}"
1,ham,ok lar joking wif u oni,ok lar joking wif u oni,"[(ok, INTJ), (lar, ADJ), (joking, NOUN), (wif, NOUN), (u, PRON), (oni, NOUN)]","{'INTJ': 1, 'ADJ': 1, 'NOUN': 3, 'PRON': 1}"
2,spam,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,"[(free, ADJ), (entry, NOUN), (2, NUM), (wkly, ADJ), (comp, NOUN), (win, NOUN), (fa, PROPN), (cup, PROPN), (final, ADJ), (tkts, NOUN), (21st, NOUN), (may, AUX), (2005, NUM), (text, NOUN), (fa, NOUN), (87121, NUM), (receive, VERB), (entry, NOUN), (questionstd, NOUN), (txt, VERB), (ratetcs, NOUN), (apply, VERB), (08452810075over18s, PROPN)]","{'ADJ': 3, 'NOUN': 10, 'NUM': 3, 'PROPN': 3, 'AUX': 1, 'VERB': 3}"
3,ham,u dun say early hor u c already say,u dun say early hor u c already say,"[(u, PRON), (dun, PROPN), (say, VERB), (early, ADJ), (hor, NOUN), (u, NOUN), (c, PROPN), (already, ADV), (say, VERB)]","{'PRON': 1, 'PROPN': 2, 'VERB': 2, 'ADJ': 1, 'NOUN': 2, 'ADV': 1}"
4,ham,nah dont think go usf life around though,nah dont think go usf life around though,"[(nah, INTJ), (do, AUX), (nt, PART), (think, VERB), (go, VERB), (usf, ADJ), (life, NOUN), (around, ADP), (though, ADV)]","{'INTJ': 1, 'AUX': 1, 'PART': 1, 'VERB': 2, 'ADJ': 1, 'NOUN': 1, 'ADP': 1, 'ADV': 1}"


### 4.2 Named Entity Recognition (NER)

**NER** identifies and classifies real-world entities in text — organizations, locations, phone numbers, monetary values, etc. Spam messages frequently reference prizes, companies, and phone numbers, making NER a valuable discriminative signal.

| NER Label | Meaning | Spam relevance |
|-----------|---------|----------------|
| `MONEY` | Monetary values | 🔴 High — *"Win $500"* |
| `CARDINAL` | Numbers | 🔴 High — *"Call 08001234"* |
| `ORG` | Companies / organizations | 🟡 Moderate |
| `DATE` / `TIME` | Temporal references | 🟡 Moderate |
| `GPE` | Countries / cities | 🟢 Low |

We reuse the same spaCy `en_core_web_sm` model loaded earlier. For each message, we extract entity text, type, and count.


In [20]:
def final_ner_summary(text):
    doc = nlp(text)
    details = [(ent.text, ent.label_) for ent in doc.ents]
    counts = dict(Counter([ent.label_ for ent in doc.ents]))
    return {"Entities": details, "Counts": counts}

data['ner_analysis'] = data['cleaned_text'].apply(final_ner_summary)

data.head(10)

,label,cleaned_text,normalized_text,pos_tags,pos_features,ner_analysis
0,ham,go jurong point crazy available bugis n great world la e buffet cine got amore wat,go jurong point crazy available bugis and great world e buffet cine got amore wat,"[(go, VERB), (jurong, PROPN), (point, NOUN), (crazy, PROPN), (available, ADJ), (bugis, PROPN), (n, ADP), (great, ADJ), (world, NOUN), (la, PROPN), (e, PROPN), (buffet, PROPN), (cine, PROPN), (got, VERB), (amore, ADV), (wat, NOUN)]","{'VERB': 2, 'PROPN': 7, 'NOUN': 3, 'ADJ': 2, 'ADP': 1, 'ADV': 1}","{'Entities': [], 'Counts': {}}"
1,ham,ok lar joking wif u oni,ok lar joking wif u oni,"[(ok, INTJ), (lar, ADJ), (joking, NOUN), (wif, NOUN), (u, PRON), (oni, NOUN)]","{'INTJ': 1, 'ADJ': 1, 'NOUN': 3, 'PRON': 1}","{'Entities': [], 'Counts': {}}"
2,spam,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s,"[(free, ADJ), (entry, NOUN), (2, NUM), (wkly, ADJ), (comp, NOUN), (win, NOUN), (fa, PROPN), (cup, PROPN), (final, ADJ), (tkts, NOUN), (21st, NOUN), (may, AUX), (2005, NUM), (text, NOUN), (fa, NOUN), (87121, NUM), (receive, VERB), (entry, NOUN), (questionstd, NOUN), (txt, VERB), (ratetcs, NOUN), (apply, VERB), (08452810075over18s, PROPN)]","{'ADJ': 3, 'NOUN': 10, 'NUM': 3, 'PROPN': 3, 'AUX': 1, 'VERB': 3}","{'Entities': [('2', 'CARDINAL'), ('21st may 2005', 'DATE'), ('87121', 'DATE')], 'Counts': {'CARDINAL': 1, 'DATE': 2}}"
3,ham,u dun say early hor u c already say,u dun say early hor u c already say,"[(u, PRON), (dun, PROPN), (say, VERB), (early, ADJ), (hor, NOUN), (u, NOUN), (c, PROPN), (already, ADV), (say, VERB)]","{'PRON': 1, 'PROPN': 2, 'VERB': 2, 'ADJ': 1, 'NOUN': 2, 'ADV': 1}","{'Entities': [], 'Counts': {}}"
4,ham,nah dont think go usf life around though,nah dont think go usf life around though,"[(nah, INTJ), (do, AUX), (nt, PART), (think, VERB), (go, VERB), (usf, ADJ), (life, NOUN), (around, ADP), (though, ADV)]","{'INTJ': 1, 'AUX': 1, 'PART': 1, 'VERB': 2, 'ADJ': 1, 'NOUN': 1, 'ADP': 1, 'ADV': 1}","{'Entities': [], 'Counts': {}}"
5,spam,freemsg hey darling 3 week word back ! id like fun still tb ok ! xxx std chgs send å£150 rcv,freemsg hey darling 3 week word back ! id like fun still tb ok ! xxx std chgs send å£150 rcv,"[(freemsg, INTJ), (hey, INTJ), (darling, NOUN), (3, NUM), (week, NOUN), (word, NOUN), (back, ADV), (!, PUNCT), (i, PRON), (d, NOUN), (like, INTJ), (fun, NOUN), (still, ADV), (tb, ADP), (ok, INTJ), (!, PUNCT), (xxx, PROPN), (std, PROPN), (chgs, PROPN), (send, VERB), (å£150, PROPN), (rcv, NOUN)]","{'INTJ': 4, 'NOUN': 6, 'NUM': 1, 'ADV': 2, 'PUNCT': 2, 'PRON': 1, 'ADP': 1, 'PROPN': 4, 'VERB': 1}","{'Entities': [('3 week', 'DATE')], 'Counts': {'DATE': 1}}"
6,ham,even brother like speak treat like aid patent,even brother like speak treat like aid patent,"[(even, ADV), (brother, NOUN), (like, ADP), (speak, VERB), (treat, NOUN), (like, ADP), (aid, NOUN), (patent, NOUN)]","{'ADV': 1, 'NOUN': 4, 'ADP': 2, 'VERB': 1}","{'Entities': [], 'Counts': {}}"
7,ham,per request melle melle oru minnaminunginte nurungu vettam set callertune caller press 9 copy friend callertune,per request melle melle oru minnaminunginte nurungu vettam set callertune caller press 9 copy friend callertune,"[(per, ADP), (request, NOUN), (melle, PROPN), (melle, PROPN), (oru, PROPN), (minnaminunginte, PROPN), (nurungu, PROPN), (vettam, PROPN), (set, PROPN), (callertune, PROPN), (caller, PROPN), (press, NOUN), (9, NUM), (copy, NOUN), (friend, NOUN), (callertune, NOUN)]","{'ADP': 1, 'NOUN': 5, 'PROPN': 9, 'NUM': 1}","{'Entities': [('melle melle', 'PERSON'), ('nurungu vettam', 'PERSON'), ('9', 'CARDINAL')], 'Counts': {'PERSON': 2, 'CARDINAL': 1}}"
8,spam,winner ! ! valued network customer selected receivea å£900 prize reward ! claim call 09061701461 claim code kl341 valid 12 hour,winner ! ! valued n

#### Visualize NER Entity Distribution

The chart below shows the most frequently occurring entity types across the full dataset, giving a high-level view of what real-world references appear most in this SMS corpus.


In [21]:
import plotly.express as px

all_labels = [label for row in data['ner_analysis'] for label, count in row['Counts'].items() for _ in range(count)]
df_plot = pd.DataFrame(Counter(all_labels).most_common(10), columns=['Type', 'Count'])

fig = px.bar(df_plot, x='Type', y='Count', title='Named Entity Analysis',
             color='Count', color_continuous_scale='Purp', text_auto=True)

fig.update_layout(coloraxis_showscale=False)
fig.show()

---

## 🔢 Section 5 — Splitting & Vectorization <a name="vectorization"></a>

### 5.1 Label Encoding & Train/Test Split

Before modeling, we:
1. **Encode labels** — map `"spam"` → `1`, `"ham"` → `0`
2. **Split data** — 80% train / 20% test with **stratified sampling** to preserve the original class ratio in both sets

Stratification is essential with imbalanced classes — without it, the test set might contain too few spam examples for meaningful evaluation.

### 5.2 Binary Bag-of-Words Vectorization

We use `CountVectorizer(binary=True)` which produces a **binary presence/absence matrix**. Each message becomes a vector where `1` = word appears, `0` = word absent.

**Why binary instead of raw counts?**
- Reduces the impact of word repetition (spam often repeats keywords aggressively)
- Simplifies probability calculations in Naive Bayes
- Works particularly well for short texts like SMS messages

> ⚠️ The vectorizer is **fit on training data only** and then applied to test data — this prevents data leakage.


In [22]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

data['label_num'] = data['label'].map({'spam': 1, 'ham': 0})

x_train_text, x_test_text, y_train, y_test = train_test_split(
    data['normalized_text'],
    data['label_num'],
    test_size=0.2,
    random_state=42,
    stratify=data['label_num']
)

vectorizer = CountVectorizer(binary=True)
x_train = vectorizer.fit_transform(x_train_text)
x_test = vectorizer.transform(x_test_text)
print(f"x_train shape: {x_train.shape}")
print(f"x_test shape : {x_test.shape}")

x_train shape: (4457, 7794)
x_test shape : (1115, 7794)


---

## 🤖 Section 6 — Model Training <a name="training"></a>

### Multinomial Naive Bayes

We train a **Multinomial Naive Bayes** classifier — a probabilistic model based on Bayes' theorem with the "naive" assumption that all features are conditionally independent given the class label.

**Why Naive Bayes for spam detection?**

| Property | Benefit |
|----------|---------|
| Simple & fast | Trains in milliseconds, even on large vocabularies |
| Works well with binary features | Matches our `CountVectorizer(binary=True)` setup |
| Proven track record | Industry standard for text classification since the 1990s |
| Handles high dimensionality naturally | Vocabulary can span tens of thousands of features |
| Uses prior probabilities | Naturally reflects class frequency from training data |

The model learns $P(\text{word} \mid \text{spam})$ and $P(\text{word} \mid \text{ham})$ for every word in the vocabulary, then uses Bayes' rule to assign the most probable class to new messages.


In [23]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(x_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


---

## 📈 Section 7 — Model Evaluation <a name="evaluation"></a>

### Metrics Explained

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| **Accuracy** | (TP + TN) / Total | Overall correct predictions |
| **Precision** | TP / (TP + FP) | Of predicted spams, how many were actually spam? |
| **Recall** | TP / (TP + FN) | Of actual spams, how many did we catch? |
| **F1-Score** | 2 · (P · R) / (P + R) | Harmonic mean of precision and recall |

> **Key trade-off:** In spam detection, **recall is more critical than precision**. Missing a phishing or malicious message is more harmful than occasionally flagging a legitimate one. A high F1 score on the spam class is the ultimate goal.


In [24]:
from sklearn.metrics import accuracy_score, classification_report
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy * 100}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

Accuracy Score: 98.20627802690582%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        Spam       0.97      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



---

## 📊 Section 8 — Results & Visualization <a name="results"></a>

### 8.1 Confusion Matrix

The confusion matrix shows the distribution of correct and incorrect predictions across both classes:

|  | Predicted Ham | Predicted Spam |
|--|:---:|:---:|
| **Actual Ham** | ✅ True Negative | ⚠️ False Positive |
| **Actual Spam** | ❌ False Negative | ✅ True Positive |

- **False Negatives** (spam → predicted ham) are the most dangerous errors — real spam slips through
- **False Positives** (ham → predicted spam) are annoying but less harmful


In [25]:
from sklearn.metrics import confusion_matrix
import plotly.express as px
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Purp',
    labels=dict(x="Predicted", y="True", color="Count"),
    x=['Ham','Spam'],
    y=['Ham','Spam'],
    title="Actual vs Predicted"
)
fig.update_layout(width=500, height=500, coloraxis_showscale=False)
fig.show()

---

### 8.2 Inference Pipeline

The `predict_spam()` function wraps the **entire preprocessing pipeline** into a single callable, ensuring the same transformations used at training time are applied at inference:

```
Raw input text
    → remove_punctuation()  → .lower()  → tokenization()
    → remove_stopwords()    → lemmatizer()  → normalize_text()
    → vectorizer.transform()   →   model.predict()
    → "SPAM" 🚨  or  "HAM" ✅
```

This training-serving consistency is critical — any mismatch between training preprocessing and inference preprocessing will silently degrade accuracy.


In [26]:
def predict_spam(text):
    text = remove_punctuation(text)
    text = text.lower()
    tokens = tokenization(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatizer(tokens)
    cleaned_text = " ".join(tokens)
    normalized = normalize_text(cleaned_text)

    vectorized_text = vectorizer.transform([normalized])
    prediction = model.predict(vectorized_text)

    return "SPAM" if prediction[0] == 1 else "HAM"


### 8.3 Evaluate on Unseen Messages

We test the model on three hand-crafted examples — two obvious spam messages and one legitimate — to verify the end-to-end pipeline works correctly on real-world input.


In [27]:
test_messages = [
    "URGENT! You have won a 1 week FREE membership. Txt CLAIM to 81010",
    "Are we still meeting for the project discussion tomorrow?",
    "Congratulations! Call 09061701461 to claim your reward."]

results = [{"Message": msg, "Prediction": predict_spam(msg)} for msg in test_messages]
pd.DataFrame(results)

,Message,Prediction
0,URGENT! You have won a 1 week FREE membership. Txt CLAIM to 81010,SPAM
1,Are we still meeting for the project discussion tomorrow?,HAM
2,Congratulations! Call 09061701461 to claim your reward.,SPAM


---

## ✅ Summary & Conclusions

### Pipeline Overview

```
spam.csv  →  Preprocessing  →  POS/NER Analysis  →  BoW Vectorization  →  Naive Bayes  →  Prediction
```

### Results Summary

| Component | Choice | Why |
|-----------|--------|-----|
| Preprocessing | NLTK (tokenize, lemmatize, stopwords) | Mature, well-tested NLP toolkit |
| POS Tagging / NER | spaCy `en_core_web_sm` | More accurate on informal SMS text |
| Vectorization | `CountVectorizer(binary=True)` | Simple and effective for Naive Bayes |
| Classifier | Multinomial Naive Bayes | Fast, probabilistic, proven for text |

### Key Takeaways

- **Selective punctuation removal** — keeping `$` and `!` preserves important spam signals
- **spaCy > NLTK** for POS/NER on short, informal, real-world text
- **Binary BoW + Naive Bayes** is a strong, interpretable baseline for spam detection
- **Stratified splitting** is essential when dealing with class imbalance